# Monolingual tokenizer analysis


## Imports and loading

In [2]:
import sys, matplotlib, pandas as pd
print(sys.executable)
print(matplotlib.__version__, pd.__version__)

/Users/sanderland/Desktop/unigram/script_bpe/.venv/bin/python
3.10.5 2.2.3


In [3]:
from script_bpe.corpus.registry import MONOLINGUAL_DATASETS, load_corpus_by_name
from script_bpe.train import load_tokenizers_for_dataset
from script_bpe.tokenizers.bpe.stats import compression_curve
import pandas as pd

/Users/sanderland/Desktop/unigram/script_bpe/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
N = 64000
num_undecodable = {}
compression_stats = []

for file in MONOLINGUAL_DATASETS:
    # BPE models
    bpe_tokenizers = load_tokenizers_for_dataset(file, n=N, model_name="bpe")
    bpe_curves = {ptok: compression_curve(tok) for ptok, tok in bpe_tokenizers.items() if tok}
    print(f"BPE: Loaded {len(bpe_curves)} tokenizers for {file}")
    num_undecodable[file] = {}

    # Determine character count using previous heuristic: initial curve value / 2 from a script pretokenizer
    ref_name = "scriptenc" if "scriptenc" in bpe_curves else ("scriptenc_cb" if "scriptenc_cb" in bpe_curves else None)
    total_char_len = None
    if ref_name is not None:
        total_char_len = bpe_curves[ref_name][0] / 2

    # Record BPE start/end where possible
    for ptok, curve in bpe_curves.items():
        if total_char_len and total_char_len > 0:
                dict(file=file, model="bpe", pretokenizer=ptok, start=curve[0] / total_char_len, end=curve[-1] / total_char_len)
            )
        # undecodable count
        num_undecodable[file][f"bpe:{ptok}"] = bpe_tokenizers[ptok].stats()["num_undecodable"]

    # Unigram models
    uni_tokenizers = load_tokenizers_for_dataset(file, n=N, model_name="unigram")
    uni_tokenizers = {ptok: tok for ptok, tok in uni_tokenizers.items() if tok}
    if uni_tokenizers:
        print(f"Unigram: Loaded {len(uni_tokenizers)} tokenizers for {file}")
    for ptok, tok in uni_tokenizers.items():
        corpus = load_corpus_by_name(file, tok.pretokenizer)
        perf = tok.corpus_performance(corpus)
        compression_stats.append(
            dict(file=file, model="unigram", pretokenizer=ptok, start=None, end=perf["tokens_per_char"]) 
        )
        num_undecodable[file][f"unigram:{ptok}"] = tok.stats()["num_undecodable"]

BPE: Loaded 7 tokenizers for eng_latn_300mb
BPE: Loaded 8 tokenizers for deu_latn_300mb
Unigram: Loaded 2 tokenizers for deu_latn_300mb
BPE: Loaded 0 tokenizers for vie_latn_300mb
BPE: Loaded 8 tokenizers for heb_hebr_300mb
BPE: Loaded 8 tokenizers for arb_arab_300mb
Unigram: Loaded 2 tokenizers for arb_arab_300mb
BPE: Loaded 8 tokenizers for rus_cyrl_300mb
BPE: Loaded 8 tokenizers for kor_hang_300mb
Unigram: Loaded 2 tokenizers for kor_hang_300mb
BPE: Loaded 8 tokenizers for hin_deva_300mb
BPE: Loaded 7 tokenizers for tha_thai_300mb
BPE: Loaded 8 tokenizers for zho_hans_300mb
BPE: Loaded 0 tokenizers for jpn_jpan_300mb
BPE: Loaded 0 tokenizers for pan_guru_300mb


In [5]:
def format_results(df, columns=None, floatfmt="{:.4f}", gradient="RdYlGn_r", relative=None):
    df = df[columns or df.columns]
    df = pd.concat([df, df.mean().rename("mean").to_frame().T])
    df_style = df.style
    if relative is not None:
        rel_val = df.div(df.median(axis=1), axis=0)
        df_style = df_style.background_gradient(
            cmap=gradient, gmap=rel_val, axis=None, vmin=1 - relative, vmax=1 + relative
        )
    elif gradient:
        df_style = df_style.background_gradient(cmap=gradient, axis=None)
    return df_style.format(floatfmt)


cdf = pd.DataFrame(compression_stats)

# BPE-only start table
bpe_start_df = cdf.query("model == 'bpe'").pivot(index="file", columns="pretokenizer", values="start")
if "bytes_gpt4" in bpe_start_df.columns:
    bpe_start_df = bpe_start_df.sort_values("bytes_gpt4", ascending=False)

display(
    format_results(
        bpe_start_df, columns=[c for c in ["bytes_gpt4", "scriptenc", "scriptenc_cb"] if c in bpe_start_df.columns]
    ).set_caption("Table 1 (Left two columns): Tokens/Char at start (BPE only)")
)

# Combined end table, model-prefixed columns
tmp = cdf.copy()
tmp["col"] = tmp["model"].fillna("bpe") + ":" + tmp["pretokenizer"]
end_df = tmp.pivot(index="file", columns="col", values="end")

nonosplit = [c for c in end_df.columns if "nosplit" not in c]
sort_col = "bpe:bytes_gpt4" if "bpe:bytes_gpt4" in end_df.columns else None
if sort_col:
    end_df = end_df.sort_values(sort_col, ascending=False)

display(
    format_results(end_df[nonosplit], relative=0.5).set_caption(
        "Table 1 (Compression) / Table 5 (Expanded): Tokens/Char after training (BPE and Unigram)"
    )
)

pretokenizer,bytes_gpt4,scriptenc,scriptenc_cb
tha_thai_300mb,2.6789,2.0000,2.0000
hin_deva_300mb,2.5134,2.0000,2.0000
kor_hang_300mb,2.3325,2.0000,2.0000
rus_cyrl_300mb,1.8130,2.0000,2.0000
arb_arab_300mb,1.7940,2.0000,2.0000
heb_hebr_300mb,1.7743,2.0000,2.0000
deu_latn_300mb,1.0171,2.0000,2.0000
eng_latn_300mb,1.0063,2.0000,2.0000
zho_hans_300mb,0.5058,2.0000,2.0000
mean,1.7150,2.0000,2.0000


col,bpe:bytes_gpt4,bpe:bytes_gpt4_cb,bpe:bytes_gpt4o,bpe:bytes_gpt4o_cb,bpe:scriptenc,bpe:scriptenc_cb,bpe:scriptenc_gpt4o,bpe:scriptenc_gpt4o_cb,unigram:scriptenc_cb,unigram:scriptenc_gpt4o_cb
hin_deva_300mb,0.4835,0.4835,0.2389,0.2387,0.2390,0.2387,0.2390,0.2387,nan,nan
kor_hang_300mb,0.3945,0.3945,0.3946,0.3945,0.3974,0.3973,0.3940,0.3942,0.4038,0.4006
tha_thai_300mb,0.3219,nan,0.2246,0.2048,0.2550,0.2364,0.2258,0.2047,nan,nan
heb_hebr_300mb,0.2459,0.2439,0.2447,0.2426,0.2469,0.2446,0.2448,0.2427,nan,nan
arb_arab_300mb,0.2324,0.2325,0.2298,0.2299,0.2328,0.2298,0.2329,0.2299,0.2308,0.2310
rus_cyrl_300mb,0.2120,0.2115,0.2120,0.2115,0.2121,0.2123,0.2111,0.2115,nan,nan
eng_latn_300mb,0.2118,0.2118,0.2114,0.2114,0.2162,0.2145,0.2127,nan,nan,nan
deu_latn_300mb,0.2007,0.2007,0.2008,0.2008,0.2066,0.2019,0.2052,0.2008,0.2013,0.2005
zho_hans_300mb,0.0939,0.5136,0.5139,0.5137,0.5612,0.5587,0.5174,0.5134,nan,nan
mean,0.2663,0.3115,0.2745,0.2720,0.2852,0.2816,0.2759,0.2795,0.2787,0.2774


In [6]:
nonosplit = [c for c in end_df.columns if "nosplit" not in c]
num_undecodable_df = pd.DataFrame.from_dict(num_undecodable, orient="index")
if "bytes_gpt4" in num_undecodable_df.columns:
    num_undecodable_df = num_undecodable_df.sort_values(by="bytes_gpt4", ascending=False)
format_results(num_undecodable_df[nonosplit], floatfmt="{:.1f}").set_caption(
    "Table 2 (Partial char): Number tokens including partial characters"
)

,bpe:bytes_gpt4,bpe:bytes_gpt4_cb,bpe:bytes_gpt4o,bpe:bytes_gpt4o_cb,bpe:scriptenc,bpe:scriptenc_cb,bpe:scriptenc_gpt4o,bpe:scriptenc_gpt4o_cb,unigram:scriptenc_cb,unigram:scriptenc_gpt4o_cb
eng_latn_300mb,186.0,121.0,177.0,120.0,6713.0,0.0,6510.0,nan,nan,nan
deu_latn_300mb,79.0,47.0,79.0,48.0,5717.0,0.0,5550.0,0.0,0.0,0.0
heb_hebr_300mb,15303.0,30.0,15510.0,30.0,16085.0,0.0,16461.0,0.0,nan,nan
arb_arab_300mb,103.0,54.0,101.0,54.0,17991.0,0.0,18109.0,0.0,0.0,0.0
rus_cyrl_300mb,66.0,30.0,66.0,30.0,107.0,0.0,112.0,0.0,nan,nan
kor_hang_300mb,1021.0,508.0,1021.0,508.0,208.0,0.0,239.0,0.0,0.0,0.0
hin_deva_300mb,310.0,119.0,162.0,64.0,255.0,0.0,215.0,0.0,nan,nan
tha_thai_300mb,180.0,nan,42831.0,104.0,22499.0,0.0,45395.0,0.0,nan,nan
zho_hans_300mb,516.0,367.0,524.0,367.0,161.0,0.0,671.0,0.0,nan,nan
mean,1973.8,159.5,6719.0,147.2,7748.4,0.0,10362.4,0.0,0.0,0.0
